# Module 12 — Inheritance, protocols and `@dataclass`

Three things, in increasing order of how much they will change what you write.
Inheritance is mostly familiar. The protocols are the part with no Java equivalent.
`@dataclass` is the one you will actually use every week.

## 1. Inheritance, and the call that is not automatic

The parent goes in brackets after the class name. What is different from Java: the
parent's `__init__` is **not** called for you.

In [ ]:
class Device:
    def __init__(self, tag):
        self.tag = tag

    def describe(self):
        return f"device {self.tag}"


class Sensor(Device):
    def __init__(self, tag, unit):
        super().__init__(tag)  # without this line, self.tag is never set
        self.unit = unit

    def describe(self):
        return super().describe() + f" ({self.unit})"


sensor = Sensor("TH-04", "C")
print(sensor.describe())
print(isinstance(sensor, Device), issubclass(Sensor, Device))

Java inserts an implicit `super()` when you do not write one, and refuses to compile
if there is no no-argument constructor to call. Python does neither. Leave the line
out and the object is simply missing whatever the parent would have set — with no
error until something reads the attribute, somewhere else.

In [ ]:
class Device:
    def __init__(self, tag):
        self.tag = tag


class Broken(Device):
    def __init__(self, tag):
        self.extra = 1  # super().__init__ forgotten


broken = Broken("TH-04")
print(vars(broken))  # no tag, and nothing complained
print(hasattr(broken, "tag"))

## 2. `super()` follows the MRO, not the parent

This is the part worth slowing down for, because the obvious reading of `super()` is
wrong. It does not mean "my parent class". It means **the next class in the method
resolution order of the object's type** — which depends on the object, not on the
class the line is written in.

With single inheritance the two are the same thing. With more than one parent they
are not.

In [ ]:
class A:
    def who(self):
        return "A"


class B(A):
    def who(self):
        return "B->" + super().who()


class C(A):
    def who(self):
        return "C->" + super().who()


class D(B, C):
    pass


print([cls.__name__ for cls in D.__mro__])

In [ ]:
# `super()` inside B.who -- which class does it reach when the object is a D?
assert D().who() == ...

`super()` in `B.who` reached **`C`**, a class `B` knows nothing about and does not
inherit from. The MRO is computed for `D`, and `super()` walks that list, so the same
line in `B` goes to `A` for a `B()` and to `C` for a `D()`.

This is what makes cooperative multiple inheritance work: every class calls
`super()`, and each one runs once, in an order the last class in the chain decides.
It is also why the mental model "super means my parent" produces bugs you cannot see
in the file you are reading.

Java has no equivalent, because a class has one superclass. The nearest thing is
default methods on interfaces, which is why Java requires you to disambiguate them by
hand.

Two practical rules that follow:

- **Read `Class.__mro__` when a call surprises you.** It is a plain tuple and it is
  the whole answer.
- **Prefer composition.** Deep hierarchies are harder here than in Java, not easier,
  precisely because `super()` is not local. Most of what Java does with an abstract
  base class, Python does with a protocol — section 3 — and with an object held as an
  attribute.

## 3. Protocols: `for`, `in`, `len` on your own type

Module 09 showed `with` working on anything with `__enter__` and `__exit__`. That was
not a special case: **every piece of syntax in Python is a call to a method with a
name in double underscores**, and implementing that method makes the syntax work on
your type. There is no interface to declare and nothing to inherit from.

In [ ]:
class Log:
    def __init__(self, readings):
        self.readings = readings

    def __len__(self):
        return len(self.readings)

    def __iter__(self):
        return iter(self.readings)

    def __contains__(self, value):
        return value in self.readings

    def __getitem__(self, index):
        return self.readings[index]


log = Log([21.7, 91.0, 23.1])

print(len(log))
print([value for value in log])  # __iter__
print(91.0 in log)  # __contains__
print(log[0], log[-1])  # __getitem__
print(max(log), sorted(log))  # anything that iterates now works

Two details worth knowing before you meet them:

**`__len__` decides truthiness.** With no `__bool__`, an object with `__len__` is
falsy when the length is zero. So an empty `Log` is `False` in an `if`, for free —
and a `Log` that has a `__len__` you did not think about that way can be falsy when
you meant it to be truthy.

In [ ]:
class Sized:
    def __init__(self, n):
        self.n = n

    def __len__(self):
        return self.n


print(bool(Sized(0)), bool(Sized(3)))


class Both:
    def __len__(self):
        return 5

    def __bool__(self):
        return False


print(bool(Both()))  # __bool__ wins where both exist

**`in` and `for` fall back.** Without `__contains__`, `in` iterates and compares.
Without `__iter__`, iteration falls back to calling `__getitem__` with 0, 1, 2 …
until `IndexError` — which is why a class with only `__getitem__` works in a `for`
loop.

In [ ]:
class OnlyGetitem:
    def __init__(self, items):
        self.items = items

    def __getitem__(self, index):
        return self.items[index]  # raises IndexError at the end, which stops the loop


only = OnlyGetitem(["a", "b"])
print([value for value in only])
print("a" in only)

The operators work the same way: `__add__` for `+`, `__lt__` for `<`, `__call__` to
make an object callable. And Python fills in the mirror image — define `__lt__` and
`a > b` works, because it asks `b < a`.

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __repr__(self):
        return f"Money({self.amount})"

    def __add__(self, other):
        return Money(self.amount + other.amount)

    def __lt__(self, other):
        return self.amount < other.amount


print(Money(1) + Money(2))
print(sorted([Money(3), Money(1)]))  # sorting needs only __lt__
print(Money(3) > Money(1))  # asks Money(1) < Money(3)

The judgement call: implement a protocol when your type **is** that thing. A `Log` is
a sequence of readings, so `len` and `for` read correctly on it. Defining `__add__` on
a type where "adding" has no obvious meaning makes code shorter and worse — the reader
now has to look up what `+` does here.

## 4. `@dataclass`

Module 11 wrote `__init__`, `__repr__` and `__eq__` by hand, three times, all of them
saying the same fields over and over. `@dataclass` writes them from the field list.

In [ ]:
from dataclasses import dataclass


@dataclass
class Reading:
    tag: str  # the annotation is not optional here -- it is how the field is declared
    celsius: float
    unit: str = "C"  # a default, and it works like a default argument


reading = Reading("TH-04", 91.0)

print(reading)  # __repr__, written for you
print(reading == Reading("TH-04", 91.0))  # __eq__, over all the fields
print(reading.celsius, reading.unit)

The annotations are what makes it work: a `@dataclass` collects the annotated names
in the class body and generates `__init__`, `__repr__` and `__eq__` over them. This is
the one place in the language where a type annotation is not merely notation.

Methods and properties go in as usual — it is an ordinary class with three methods
already written.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Log:
    tag: str
    readings: list[float] = field(default_factory=list)  # NOT `= []`

    @property
    def highest(self):
        return max(self.readings) if self.readings else None

    def add(self, value):
        self.readings.append(value)
        return self


log = Log("TH-04").add(91.0).add(23.1)
print(log)
print(log.highest)
print(Log("TH-09"), Log("TH-01"))  # each gets its own list

`= []` as a dataclass default is **an error**, not a trap:

In [ ]:
from dataclasses import dataclass

try:

    @dataclass
    class Bad:
        items: list = []

except ValueError as err:
    print("ValueError:", err)

That is module 04's mutable default argument, and this time the language refuses.
`field(default_factory=list)` says "call this to make the default", which is exactly
the fix module 04 arrived at by hand.

And the thread from modules 06 and 11 closes. A `@dataclass` writes `__eq__`, so —
just as in module 11 — it is **unhashable**.

In [ ]:
from dataclasses import dataclass


@dataclass
class Reading:
    tag: str
    celsius: float


try:
    {Reading("TH-04", 91.0)}
    outcome = "hashable"
except TypeError:
    outcome = "TypeError"

assert outcome == ...

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)  # immutable, and therefore hashable
class Reading:
    tag: str
    celsius: float


reading = Reading("TH-04", 91.0)
print(hash(reading) is not None)
print({reading, Reading("TH-04", 91.0)})  # equal, so the set keeps one

try:
    reading.celsius = 0.0
except Exception as err:
    print(type(err).__name__, "-", err)

`frozen=True` is the answer to module 06's rule: a key must not change while it is a
key, so a class that cannot change is safe to hash. It is worth reaching for by
default on anything you use as a record — a value that cannot be modified behind your
back is easier to reason about, and mutating one is usually a bug anyway.

The other arguments worth knowing: `order=True` writes `<`, `<=`, `>`, `>=` over the
fields in order, and `slots=True` trades the ability to add attributes for less memory
per object.

In [ ]:
from dataclasses import asdict, dataclass, fields


@dataclass(frozen=True, order=True)
class Reading:
    celsius: float
    tag: str = "?"


print(sorted([Reading(91.0, "TH-04"), Reading(21.7, "TH-01")]))
print(asdict(Reading(91.0, "TH-04")))  # straight into json.dumps, module 08
print([f.name for f in fields(Reading)])

`asdict` is the reason a dataclass sits well at the edge of a program: a record with
checked field names inside, a plain dict on the way out to JSON.

Which answers the question module 06 left open — dict or class. **A dict when the
keys are data; a frozen dataclass when they are your design.** The dataclass costs
three lines and gives you the names checked by an editor, a readable repr in every
traceback, equality that means what you meant, and `asdict` when you need the dict
back.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run.

Module 13 is the last of Part 3: iterators and generators — `yield`, and the protocol
behind the `for` loop you have been using since module 03.